In [ ]:
import pandas as pd
from itertools import product

from torch import candidate

In [ ]:
ir_file = 'YAGOrules.txt-100'
relations_file = '../datasets/YAGO3-10/relations.ttl'
nice_relations_file = '../datasets/YAGO3-10/relations_formatted.ttl'

In [ ]:
rules = []

def triple_split(s):
    first_part = s.split('(', 1)[0]
    second_part = s.split('(', 1)[1]
    chars = second_part.strip(')').split(',')
    return first_part, chars[0], chars[1]


with open(ir_file) as fIR:
    for line in fIR.readlines():
        rule = line.strip('\n').split('\t')[-1]
        ht,bt = rule.split(' <= ')
        bt = bt.replace('),',')')
        h = triple_split(ht)
        b = [triple_split(triple) for triple in bt.split(' ')]
       # b= [atom.replace('),',')') for atom in b]

        rules.append((h,b))

In [ ]:
newrules = []
for rule in rules:

    body_props = set([p[0] for p in rule[1]])
    extended= (rule[0], rule[1], body_props)
    newrules.append(extended)

In [ ]:
with open(relations_file, 'r') as infile, open(nice_relations_file, 'w') as outfile:
    for line in infile:
        # Use .split() with no arguments to split by any whitespace (spaces, tabs, newlines)
        # and handle multiple whitespaces between strings.


        parts = line.strip().split()
        if len(parts) < 4 or '"' in line:
            continue
        # Ensure there are at least 4 parts to avoid errors
        if line[0] != '#' and line[1] != '#' and len(parts) >= 4:
            # Take the first four parts and join them with a tab
            formatted_line = '\t'.join(parts[:4])
            outfile.write(formatted_line + '\n')
        else:
            # Optional: handle lines that don't have 4 parts
            # For example, you could write them as-is or skip them.
            # Here, we'll write the line as-is, just in case.
            outfile.write(line)

In [ ]:
relations = pd.read_csv(nice_relations_file, sep='\t',comment='#', skiprows=15, names=['s', 'r','o', '.'])

In [ ]:
relations

In [ ]:
dr =relations[(relations['r'] == 'rdfs:domain') | (relations['r'] == 'rdfs:domain')]

superClass_mapping = {
    '<wordnet_airport_102692232>' : '<yagoGeoEntity>',
    'yagoPermanentlyLocatedEntity' : '<yagoGeoEntity>',
    '<wordnet_movie_106613686>':'<wordnet_artifact_100021939>',
    '<wordnet_language_106282651>' : '<wordnet_abstraction_100002137>',
    '<wordnet_actor_109765278>': '<wordnet_person_100007846>',
    '<wordnet_country_108544813>' : '<yagoGeoEntity>',
    '<wordnet_location_100027167>': '<yagoGeoEntity>',
    '<wordnet_event_100029378>': '<wordnet_abstraction_100002137>',
}

dr['superO'] = dr['o'].apply(lambda x: superClass_mapping.get(x) if x in superClass_mapping.keys() else x)

In [ ]:
dr

In [ ]:
def get_restr(prop, domain=True):
    if domain:
        res = dr[(dr['s'] == '<'+prop+'>') & (dr['r'] == 'rdfs:domain')]
        if len(res)> 0:
            return res['superO'].item()
        else:
            return '<owl:Thing>'
    else:
        res= dr[(dr['s'] == '<'+prop+'>') & (dr['r'] == 'rdfs:range')]
        if len(res)> 0:
            return res['superO'].item()
        else:
            return '<owl:Thing>'


In [ ]:
r1

In [ ]:
candidates=[]
disjoints = ['<wordnet_abstraction_100002137>','<wordnet_artifact_100021939>','<wordnet_building_102913152>','<wordnet_organization_108008335>','<wordnet_person_100007846>','<wordnet_physical_entity_100001930>', '<yagoGeoEntity>']
one_exception = ['<yagoGeoEntity>','<wordnet_organization_108008335>']

for r1,r2 in product(newrules, repeat=2):
    p1 = r1[0][0]
    p2 = r2[0][0]
    dr1=get_restr(p1)
    rr1=get_restr(p1,False)
    dr2=get_restr(p2)
    rr2=get_restr(p2,False)
    if r1[2].intersection(r2[2]):
        if dr1 != dr2 and dr1 in disjoints and dr2 in disjoints and not (dr1 in one_exception and dr2 in one_exception):
            candidates.append((r1,r2))
        elif dr1 != rr2 and dr1 in disjoints and rr2 in disjoints and not (dr1 in one_exception and rr2 in one_exception):
            candidates.append((r1,r2))
        elif dr2 != rr1 and dr2 in disjoints and rr1 in disjoints and not (dr2 in one_exception and rr1 in one_exception):
            candidates.append((r1,r2))
        elif dr2 != rr2 and dr2 in disjoints and rr2 in disjoints and not (dr2 in one_exception and rr2 in one_exception):
            candidates.append((r1,r2))

In [ ]:
r1[2]

In [ ]:
candidates

In [ ]:
len(candidates)

In [ ]:
candidates